In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("UberAnalyticsProject") \
    .getOrCreate()



In [26]:
df =spark.read.parquet('/content/travel.parquet')

df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [27]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



#Data Cleaning

In [31]:
from pyspark.sql.functions import col
clean_df=df.filter(col("trip_distance")>0)
clean_df=clean_df.filter(col("fare_amount")>0)
clean_df=clean_df.filter(col("passenger_count")>0)
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

Feature Engineering

In [36]:
#Trip Duration
from pyspark.sql.functions import unix_timestamp
clean_df=clean_df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime")))/60
)

In [37]:
#Extract Hour of Ride
from pyspark.sql.functions import hour
clean_df=clean_df.withColumn("pickup_hour",hour(col("tpep_pickup_datetime")))

# Analytics 1


In [38]:
# Busiest Hours
from pyspark.sql.functions import count
busy_hours=clean_df.groupBy("pickup_hour").agg(count("*").alias("ride_count")) \
.orderBy("ride_count",ascending=False)
busy_hours.show()

+-----------+----------+
|pickup_hour|ride_count|
+-----------+----------+
|         17|    206976|
|         18|    205987|
|         16|    190748|
|         15|    188209|
|         14|    178804|
|         19|    174654|
|         13|    165943|
|         21|    161521|
|         20|    157909|
|         12|    157095|
|         11|    144633|
|         22|    134143|
|         10|    133610|
|          9|    122244|
|          8|    105864|
|         23|     94346|
|          7|     74158|
|          0|     65124|
|          1|     43681|
|          6|     34876|
+-----------+----------+
only showing top 20 rows


In [39]:
#Top Pickup Locations
from pyspark.sql.functions import count
pickup_locations=clean_df.groupBy("PULocationID").agg(count("*").alias("trip_count")) \
.orderBy("trip_count",ascending=False)
pickup_locations.show()


+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         237|    148137|
|         161|    146843|
|         236|    137941|
|         132|    133734|
|         186|    108204|
|         230|    107546|
|         162|    105098|
|         142|     97611|
|         138|     85431|
|         163|     84672|
|         239|     81858|
|         170|     79561|
|         234|     78068|
|          68|     75309|
|          48|     70222|
|         141|     70017|
|         140|     63340|
|         164|     61333|
|          79|     59647|
|         249|     59585|
+------------+----------+
only showing top 20 rows


In [40]:
#Daily Revenue
from pyspark.sql.functions import sum,to_date
revenue_df=clean_df.withColumn(
    "date",
    to_date(col("tpep_pickup_datetime"))
)

daily_revenue=revenue_df.groupBy("date").agg(sum("fare_amount").alias("total_revenue"))
daily_revenue.show()

+----------+------------------+
|      date|     total_revenue|
+----------+------------------+
|2025-01-09|  1761781.63000002|
|2025-01-14|1741481.8500000248|
|2025-01-18| 1547606.150000012|
|2025-01-17|1733388.6799999974|
|2025-01-13| 1539655.090000021|
|2024-12-31|401.59999999999997|
|2025-01-10| 1684731.550000051|
|2025-01-08| 1640261.290000014|
|2025-01-19|1300654.2400000056|
|2025-01-12|1515265.8800000076|
|2025-01-20|2206107.0800000285|
|2025-01-16|1955383.7200000184|
|2025-01-22| 1694042.080000008|
|2025-01-01|1461570.1900000132|
|2025-01-05| 1451815.420000012|
|2025-01-23|1856621.6299999983|
|2025-01-02|1583667.1900000041|
|2025-01-07|1559803.8700000278|
|2025-01-21| 1636616.189999995|
|2025-01-06|1395951.3900000113|
+----------+------------------+
only showing top 20 rows


In [41]:
#Average Trip Distance
from pyspark.sql.functions import avg
avg_distance=clean_df.agg(avg("trip_distance").alias("average_distance"))
avg_distance.show()


+------------------+
|  average_distance|
+------------------+
|3.2305888417318296|
+------------------+



In [42]:
busy_hours.write.mode("overwrite").parquet("/content/busy_hours")
pickup_locations.write.mode("overwrite").parquet("/content/pickup_locations")
daily_revenue.write.mode("overwrite").parquet("/content/daily_revenue")

#SQL Version

In [43]:
clean_df.createOrReplaceTempView("trips")

In [44]:
spark.sql("""
Select pickup_hour,count(*) as trip_count
from trips
group by pickup_hour
order by trip_count desc
""").show()

+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         17|    206976|
|         18|    205987|
|         16|    190748|
|         15|    188209|
|         14|    178804|
|         19|    174654|
|         13|    165943|
|         21|    161521|
|         20|    157909|
|         12|    157095|
|         11|    144633|
|         22|    134143|
|         10|    133610|
|          9|    122244|
|          8|    105864|
|         23|     94346|
|          7|     74158|
|          0|     65124|
|          1|     43681|
|          6|     34876|
+-----------+----------+
only showing top 20 rows


#Performance Optimization

In [45]:
clean_df.repartition("pickup_hour")

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, trip_duration_minutes: double, pickup_hour: int]

In [46]:
clean_df.cache()

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, trip_duration_minutes: double, pickup_hour: int]